In [1]:
!pip install -q joblib

import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.multioutput import MultiOutputClassifier

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

from sklearn.metrics import f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [2]:
uploaded = files.upload()

file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name)

print("Shape:", df.shape)
display(df.head())
print("\nColumns:", df.columns.tolist())

Saving maternal.csv to maternal.csv
Shape: (8000, 16)


,age_years,gestational_age_weeks,systolic_bp_mmHg,heart_rate_bpm,vaginal_bleeding,severe_headache_or_vision_issues,abdominal_pain_severity_0_10,fetal_movement,fever_present,seizures,previous_complications,hemoglobin_g_dL,edema,duration_days,clinical_disposition,severity_score
0,31,30,115.6,71.2,No,No,1,Normal,No,No,Yes,11.2,NaN,0,Treat + monitor,Low
1,40,41,132.3,109.4,No,Yes,4,Normal,No,No,No,9.9,Mild,4,Treat + monitor,Medium
2,28,31,104.5,82.2,No,No,0,Normal,No,No,No,10.2,NaN,0,Treat locally,Low
3,15,4,104.1,97.0,No,No,3,Normal,No,No,No,9.3,NaN,2,Treat + monitor,Low
4,29,14,65.2,122.7,No,Yes,9,Normal,No,No,No,6.5,Generalized,4,Stabilize + refer,High



Columns: ['age_years', 'gestational_age_weeks', 'systolic_bp_mmHg', 'heart_rate_bpm', 'vaginal_bleeding', 'severe_headache_or_vision_issues', 'abdominal_pain_severity_0_10', 'fetal_movement', 'fever_present', 'seizures', 'previous_complications', 'hemoglobin_g_dL', 'edema', 'duration_days', 'clinical_disposition', 'severity_score']


In [3]:
expected_cols = [
    'age_years',
    'gestational_age_weeks',
    'systolic_bp_mmHg',
    'heart_rate_bpm',
    'vaginal_bleeding',
    'severe_headache_or_vision_issues',
    'abdominal_pain_severity_0_10',
    'fetal_movement',
    'fever_present',
    'seizures',
    'previous_complications',
    'hemoglobin_g_dL',
    'edema',
    'duration_days',
    'clinical_disposition',
    'severity_score'
]

missing = [c for c in expected_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

df = df[expected_cols].drop_duplicates().reset_index(drop=True)

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()

df.head()

,age_years,gestational_age_weeks,systolic_bp_mmHg,heart_rate_bpm,vaginal_bleeding,severe_headache_or_vision_issues,abdominal_pain_severity_0_10,fetal_movement,fever_present,seizures,previous_complications,hemoglobin_g_dL,edema,duration_days,clinical_disposition,severity_score
0,31,30,115.6,71.2,No,No,1,Normal,No,No,Yes,11.2,nan,0,Treat + monitor,Low
1,40,41,132.3,109.4,No,Yes,4,Normal,No,No,No,9.9,Mild,4,Treat + monitor,Medium
2,28,31,104.5,82.2,No,No,0,Normal,No,No,No,10.2,nan,0,Treat locally,Low
3,15,4,104.1,97.0,No,No,3,Normal,No,No,No,9.3,nan,2,Treat + monitor,Low
4,29,14,65.2,122.7,No,Yes,9,Normal,No,No,No,6.5,Generalized,4,Stabilize + refer,High


In [4]:
clinical_order = [
    'Treat locally',
    'Treat + monitor',
    'Stabilize + refer',
    'Emergency referral'
]

severity_order = ['Low','Medium','High']

clinical_map = {v:i for i,v in enumerate(clinical_order)}
severity_map = {v:i for i,v in enumerate(severity_order)}

inv_clinical = {i:v for v,i in clinical_map.items()}
inv_severity = {i:v for v,i in severity_map.items()}

X = df.drop(['clinical_disposition','severity_score'], axis=1)

y = pd.DataFrame({
    'clinical_disposition': df['clinical_disposition'].map(clinical_map),
    'severity_score': df['severity_score'].map(severity_map)
})

In [5]:
stratify_key = df['clinical_disposition'] + "_" + df['severity_score']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=stratify_key
)

In [6]:
num_cols = [
    'age_years',
    'gestational_age_weeks',
    'systolic_bp_mmHg',
    'heart_rate_bpm',
    'abdominal_pain_severity_0_10',
    'hemoglobin_g_dL',
    'duration_days'
]

cat_cols = [
    'vaginal_bleeding',
    'severe_headache_or_vision_issues',
    'fetal_movement',
    'fever_present',
    'seizures',
    'previous_complications',
    'edema'
]

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])

In [7]:
models = {
    "logistic": LogisticRegression(max_iter=3000),
    "rf": RandomForestClassifier(n_estimators=300, random_state=42),
    "extra": ExtraTreesClassifier(n_estimators=400, random_state=42)
}

pipelines = {}

for name, model in models.items():
    pipelines[name] = Pipeline([
        ('prep', preprocessor),
        ('model', MultiOutputClassifier(model))
    ])

In [8]:
trained = {}

for name, pipe in pipelines.items():
    print("Training:", name)
    pipe.fit(X_train, y_train)
    trained[name] = pipe

Training: logistic
Training: rf
Training: extra


In [9]:
def evaluate(model, name):
    pred = model.predict(X_test)

    y_disp = y_test['clinical_disposition']
    y_sev = y_test['severity_score']

    print("\n====", name, "====")

    print("Disposition F1:", f1_score(y_disp, pred[:,0], average='macro'))
    print("Severity F1   :", f1_score(y_sev, pred[:,1], average='macro'))

    print("\nDisposition Report")
    print(classification_report(y_disp, pred[:,0], target_names=clinical_order))

    print("\nSeverity Report")
    print(classification_report(y_sev, pred[:,1], target_names=severity_order))

In [10]:
scores = []

for name, model in trained.items():
    evaluate(model, name)

    pred = model.predict(X_test)

    f1_disp = f1_score(y_test['clinical_disposition'], pred[:,0], average='macro')
    f1_sev = f1_score(y_test['severity_score'], pred[:,1], average='macro')

    score = 0.65*f1_disp + 0.35*f1_sev
    scores.append((name, score))

scores = sorted(scores, key=lambda x: x[1], reverse=True)
scores


==== logistic ====
Disposition F1: 0.8345743472159333
Severity F1   : 0.8869200693478497

Disposition Report
                    precision    recall  f1-score   support

     Treat locally       0.89      0.93      0.91       400
   Treat + monitor       0.83      0.80      0.81       400
 Stabilize + refer       0.74      0.80      0.77       400
Emergency referral       0.89      0.81      0.85       400

          accuracy                           0.83      1600
         macro avg       0.84      0.83      0.83      1600
      weighted avg       0.84      0.83      0.83      1600


Severity Report
              precision    recall  f1-score   support

         Low       0.96      0.96      0.96       683
      Medium       0.81      0.84      0.82       450
        High       0.89      0.86      0.88       467

    accuracy                           0.90      1600
   macro avg       0.89      0.89      0.89      1600
weighted avg       0.90      0.90      0.90      1600


==== rf 

[('rf', 0.9707561169395013),
 ('extra', 0.9303642200487177),
 ('logistic', 0.852895349962104)]

In [11]:
best_name = scores[0][0]
best_model = trained[best_name]

print("Best model:", best_name)

Best model: rf


In [12]:
bundle = {
    "model": best_model,
    "features": X.columns.tolist(),
    "inv_clinical": inv_clinical,
    "inv_severity": inv_severity
}

joblib.dump(bundle, "maternal_model.joblib")

from google.colab import files
files.download("maternal_model.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
sample = {
    'age_years': 28,
    'gestational_age_weeks': 32,
    'systolic_bp_mmHg': 160,
    'heart_rate_bpm': 110,
    'vaginal_bleeding': 'No',
    'severe_headache_or_vision_issues': 'Yes',
    'abdominal_pain_severity_0_10': 7,
    'fetal_movement': 'Reduced',
    'fever_present': 'No',
    'seizures': 'No',
    'previous_complications': 'Yes',
    'hemoglobin_g_dL': 9.5,
    'edema': 'Yes',
    'duration_days': 2
}

sample_df = pd.DataFrame([sample])

pred = best_model.predict(sample_df)[0]

print("Clinical:", inv_clinical[int(pred[0])])
print("Severity:", inv_severity[int(pred[1])])

Clinical: Emergency referral
Severity: High
